In [5]:
import os
import boto3
from dotenv import load_dotenv

# Configurações do Bucket
NOME_DO_BUCKET = 'treinamento2-2026.0'
REGIAO = 'us-east-2'
PASTA_DESTINO = 'bases_baixadas'

def baixar_todas_as_bases():
    # 1. Carregar credenciais
    load_dotenv()
    aws_id = os.getenv('AWS_ACCESS_KEY_ID')
    aws_secret = os.getenv('AWS_SECRET_ACCESS_KEY')

    if not aws_id or not aws_secret:
        print("❌ Erro: Credenciais não encontradas no ficheiro .env")
        return

    # 2. Inicializar cliente S3
    s3_client = boto3.client(
        's3',
        region_name=REGIAO,
        aws_access_key_id=aws_id,
        aws_secret_access_key=aws_secret
    )

    # 3. Criar a pasta de destino local se ela não existir
    if not os.path.exists(PASTA_DESTINO):
        os.makedirs(PASTA_DESTINO)
        print(f"📁 Pasta '{PASTA_DESTINO}' criada.")

    try:
        # 4. Listar todos os objetos no bucket
        print(f"🔍 Procurando ficheiros no bucket '{NOME_DO_BUCKET}'...")
        response = s3_client.list_objects_v2(Bucket=NOME_DO_BUCKET)

        if 'Contents' not in response:
            print("⚠️ O bucket está vazio.")
            return

        # 5. Iterar e baixar cada ficheiro
        print(f"🚀 Iniciando download de {len(response['Contents'])} ficheiros...\n")
        
        for item in response['Contents']:
            nome_arquivo = item['Key']
            
            # Pular se for uma "pasta" (objetos que terminam em /)
            if nome_arquivo.endswith('/'):
                continue
                
            caminho_local = os.path.join(PASTA_DESTINO, nome_arquivo)
            
            # Criar subpastas locais se o arquivo estiver dentro de uma "pasta" no S3
            subpasta_local = os.path.dirname(caminho_local)
            if not os.path.exists(subpasta_local):
                os.makedirs(subpasta_local)

            print(f"⬇️ Baixando: {nome_arquivo}...")
            s3_client.download_file(NOME_DO_BUCKET, nome_arquivo, caminho_local)
        
        print(f"\n✅ Todos os ficheiros foram baixados com sucesso na pasta '{PASTA_DESTINO}'!")

    except Exception as e:
        print(f"❌ Ocorreu um erro: {e}")

if __name__ == "__main__":
    baixar_todas_as_bases()

📁 Pasta 'bases_baixadas' criada.
🔍 Procurando ficheiros no bucket 'treinamento2-2026.0'...
🚀 Iniciando download de 6 ficheiros...

⬇️ Baixando: base_bens.csv...
⬇️ Baixando: base_financeiro.csv...
⬇️ Baixando: base_infos_pessoais.csv...
⬇️ Baixando: base_regional.csv...
⬇️ Baixando: base_scores.csv...
⬇️ Baixando: base_target.csv...

✅ Todos os ficheiros foram baixados com sucesso na pasta 'bases_baixadas'!
